# Phase 6 - Abstractive summarization with Arch 1 - GPT-2 style decoder (word-level tokenizer)

Starts from the Phase-1 model (12-layer GPT-2 style decoder (111M), learned positions, word-level lowercased vocabulary.) and writes a summary of each judgment.
Metric: BERTScore F1 (plus ROUGE-1 / ROUGE-L). The model reads `[source] <eos> [summary]` and is trained only on the summary tokens; at test time it continues after `<eos>` with a KV cache (added here - Phase 1 never needed one).

**What the data asks for.** Reference summaries are long (median ~640 words, 90% under 1,500) and about **4.6 times
shorter than their document**; documents run from ~1,300 to 140,000 words.
* **Source selection**: 1,000 tokens - 40% from the opening of the judgment (parties, facts) plus three evenly spaced
  pieces of the rest, the last ending at the final word, so the whole case is represented.
* **Length control**: each summary's length budget follows its document's length (document words / 4.6, clipped to
  150-900 words); generation has to reach 85% of it and stops by 115% - length matters a lot for BERTScore and ROUGE.
* **Redundancy control**: greedy decoding with a repetition penalty (Keskar et al. 2019) and no repeated 4-grams.

**Experiments** (each is its own submission; all from the Phase-6 list)

| Experiment | What trains |
|---|---|
| `full` | every weight - strong reference and the distillation teacher |
| `lorma` | **LoRMA** (compulsory): `(alpha/r BA + I) W x` on the attention projections (Bihany, Patel & Modi 2025) |
| `lora` | LoRA on the same projections (Hu et al. 2021) |
| `houlsby` | Houlsby bottleneck adapters after the attention and feed-forward outputs (Houlsby et al. 2019) |
| `prompt` | prompt tuning: 16 learned virtual tokens in front of the source (Lester et al. 2021) |
| `kd` | word-level distillation (Hinton et al. 2015; Kim & Rush 2016): a half-depth student learns the fine-tuned `full` model's next-token distributions |

There is no dev set, so 150 training documents are held out: teacher-forced loss on them picks the best checkpoint,
and 40 of them are actually summarised to report ROUGE-1 / ROUGE-L and length. Every experiment trains for a fixed
**time budget** (`TRAIN_MINUTES`).

**Kaggle settings:** GPU T4 x2, Internet on, secret `HF` holding your Hugging Face token. Then Run All.

## 1. Setup

In [ ]:
import os, sys, json, glob, shutil, time, gc
import numpy as np
import torch
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

WORK_DIR = "/kaggle/working/phase6_arch1"
os.makedirs(WORK_DIR, exist_ok=True)
sys.path.insert(0, WORK_DIR)
os.chdir(WORK_DIR)

print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## 2. Configuration
Everything you might want to change lives here.

In [ ]:
%%writefile /kaggle/working/phase6_arch1/config.py
"""Every knob of this Phase-6 (abstractive summarization) notebook."""
import os
import json

ARCH = "arch1"
WORK_DIR = f"/kaggle/working/phase6_{ARCH}"
DATA_DIR = os.path.join(WORK_DIR, "data")
CACHE_DIR = os.path.join(WORK_DIR, "phase1")        # Phase-1 files downloaded from Hugging Face land here
OUT_DIR = os.path.join(WORK_DIR, "runs")             # one folder per experiment

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
RAW = {"train": os.path.join(DATA_DIR, "Phase_6_Summ", "Train", "train.json"),
       "test": os.path.join(DATA_DIR, "Phase_6_Summ", "Test", "test.json")}

# ------------------------------------------------------------------ the Phase-1 model
# Looked up in this order: an attached Kaggle dataset whose path mentions "arch1", then these HF repos.
PHASE1_REPOS = ["D-240357/legal-gpt2-decoder"]
WEIGHT_FILES = ["model_state_dict.pt"]
CONFIG_FILES = ["config.json"]
TOKENIZER_FILES = ["custom_tokenizer.json"]
TOKENIZER_REPOS = PHASE1_REPOS
KAGGLE_HINT = "arch1"

SUM_REPO = "D-240357/legal-sum-arch1"           # fine-tuned checkpoints, results and submission zips go here

# ------------------------------------------------------------------ experiments
# Each one is a separate submission. "full" is the strong reference and the distillation teacher;
# "kd" waits for it to finish. "pfeiffer" (adapters after the feed-forward block only) is available too.
METHODS = ["full", "lorma", "lora", "houlsby", "prompt", "kd"]
KD_TEACHER = "full"
TRAIN_MINUTES = 40             # wall-clock training budget per experiment (the LR schedule ends exactly on it)
MAX_EPOCHS = 3

# ------------------------------------------------------------------ input and output lengths
SRC_LEN = 1000                 # source tokens the model reads
HEAD_FRAC = 0.4                # 40% from the opening of the judgment (parties, facts) ...
N_SPREAD = 3                   # ... and 3 evenly spaced pieces of the rest, the last ending at the final token
TGT_LEN = 1024                # summary tokens used for training (90% of references are shorter than this)
CHAR_CAP = 400_000             # text beyond this is not even tokenized
MAX_NEW_TOKENS = 1024         # hard ceiling at generation time (the model's context limit)
LENGTH_RATIO = "auto"          # summary words = document words / ratio (auto = training median, ~4.6) ...
MIN_SUMMARY_WORDS = 150        # ... clipped to these bounds
MAX_SUMMARY_WORDS = 900
LENGTH_SLACK = 0.15            # generation must reach 85% of the target length and stops by 115%

# ------------------------------------------------------------------ decoding
REPETITION_PENALTY = 1.15      # Keskar et al. 2019
NO_REPEAT_NGRAM = 4            # never emit the same 4-gram twice
SENTENCE_CASE = True         # arch 1 has a lower-case vocabulary: re-capitalise sentence starts
GEN_BATCH = 8

# ------------------------------------------------------------------ optimisation
MICRO_DOCS = 2
GRAD_ACCUM = 4
WARMUP_STEPS = 50
MIN_LR_RATIO = 0.1
WEIGHT_DECAY = 0.01
LABEL_SMOOTHING = 0.05
FULL_LR = 3e-5
ADAPTER_LR = 5e-4
ADAPTER_RANK = 8
LORA_ALPHA = 16
LORMA_ALPHA = 8
ADAPTER_DROPOUT = 0.05
ADAPTER_TARGETS = ["W_qkv", "W_o"]       # LoRA / LoRMA: attention projections
HOULSBY_SITES = ["W_o", "down"]         # after the attention output and the feed-forward output
PFEIFFER_SITES = ["down"]       # after the feed-forward output only
ADAPTER_BOTTLENECK = 64
PROMPT_LEN = 16
PROMPT_LR = 1e-2

KD_TEMPERATURE = 2.0
KD_ALPHA = 0.5                 # weight of the reference-summary loss; the rest goes to matching the teacher
STUDENT_LAYERS = [0, 2, 4, 6, 8, 10]

# ------------------------------------------------------------------ evaluation
DEV_DOCS = 150                 # held out from train (there is no official dev set)
DEV_LOSS_DOCS = 100            # teacher-forced loss on these picks the best checkpoint
DEV_GEN_DOCS = 40              # these are actually summarised to measure ROUGE and length
EVAL_EVERY_STEPS = 150
TRAIN_DOC_LIMIT = None         # set to e.g. 200 (and TRAIN_MINUTES = 3) for a quick smoke test
PREP_WORKERS = max(1, os.cpu_count() or 2)

SEED = 13

_override = os.environ.get("P6_CFG_OVERRIDE")
if _override:
    globals().update(json.loads(_override))

## 3. Code
The Phase-1 tokenizer and model (unchanged, so the trained weights load), plus the summarization pieces.

In [ ]:
%%writefile /kaggle/working/phase6_arch1/tokenizer.py
"""Arch 1's word-level tokenizer (exactly as in Phase 1) plus an offset-aware encoder for NER."""
import re
import json
from collections import Counter


class CustomTokenizer:
    def __init__(self, vocab_size=32000):
        self.vocab_size = vocab_size
        self.special_tokens = ["<pad>", "<unk>", "<bos>", "<eos>"]
        self.word2id = {w: i for i, w in enumerate(self.special_tokens)}
        self.id2word = {i: w for i, w in enumerate(self.special_tokens)}
        self.pad_id = 0
        self.unk_id = 1
        self.bos_id = 2
        self.eos_id = 3
        self.counter = Counter()

    def _tokenize_text(self, text):
        return re.findall(r"\w+|[^\w\s]", text, re.UNICODE)

    def encode(self, text, add_special_tokens=False):
        ids = [self.word2id.get(tok, self.unk_id) for tok in self._tokenize_text(text.lower())]
        if add_special_tokens:
            ids = [self.bos_id] + ids + [self.eos_id]
        return ids

    def decode(self, ids, skip_special_tokens=True):
        words = []
        for i in ids:
            if skip_special_tokens and i < len(self.special_tokens):
                continue
            words.append(self.id2word.get(i, "<unk>"))
        out = " ".join(words)
        out = re.sub(r"\s+([.,!?;\)])", r"\1", out)
        out = re.sub(r"(\()\s+", r"\1", out)
        return out

    def load(self, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        self.word2id = data["word2id"]
        self.id2word = {int(k): v for k, v in data["id2word"].items()}
        self.vocab_size = len(self.word2id)


# Phase 1 used \w+ which glues letters and digits ("Pradesh1", "ENT08"). Splitting them lifts the share of
# gold spans that start and end on a token boundary; both halves are ordinary vocabulary words.
NER_WORD_RE = re.compile(r"[^\W\d_]+|\d+|_+|[^\w\s]", re.UNICODE)


class NerTokenizer(CustomTokenizer):
    split_digits = True

    def encode_with_offsets(self, text):
        splitter = NER_WORD_RE if self.split_digits else re.compile(r"\w+|[^\w\s]", re.UNICODE)
        ids, starts, ends = [], [], []
        for m in splitter.finditer(text):
            ids.append(self.word2id.get(m.group().lower(), self.unk_id))
            starts.append(m.start())
            ends.append(m.end())
        return ids, starts, ends

In [ ]:
%%writefile /kaggle/working/phase6_arch1/phase1_model.py
"""
Arch 1 (GPT-2 style decoder) exactly as trained in Phase 1, so the saved weights load unchanged.
Phase 2 adds one switch: `causal`, so attention can look right as well as left while fine-tuning for NER.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_head, dropout, max_len):
        super().__init__()
        self.n_head = n_head
        self.head_dim = d_model // n_head
        self.W_qkv = nn.Linear(d_model, 3 * d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.resid_drop = nn.Dropout(dropout)
        self.dropout_p = dropout
        self.causal = True

    def forward(self, x, key_mask=None):
        B, T, D = x.shape
        qkv = self.W_qkv(x).reshape(B, T, 3, self.n_head, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        p = self.dropout_p if self.training else 0.0
        if key_mask is not None:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=key_mask, dropout_p=p)
        else:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=self.causal, dropout_p=p)
        out = out.permute(0, 2, 1, 3).reshape(B, T, D)
        return self.resid_drop(self.W_o(out))


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.up = nn.Linear(d_model, d_ff)
        self.down = nn.Linear(d_ff, d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.drop(self.down(F.gelu(self.up(x))))


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_head, d_ff, dropout, max_len):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_head, dropout, max_len)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff, dropout)

    def forward(self, x, key_mask=None):
        x = x + self.attn(self.ln1(x), key_mask)
        x = x + self.ffn(self.ln2(x))
        return x


class GPT2LM(nn.Module):
    def __init__(self, vocab_size, d_model, n_head, n_layer, d_ff, dropout, max_len):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.emb_drop = nn.Dropout(dropout)
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, n_head, d_ff, dropout, max_len)
            for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.token_emb.weight

    def forward(self, x):
        B, T = x.shape
        positions = torch.arange(T, device=x.device).unsqueeze(0)
        h = self.emb_drop(self.token_emb(x) + self.pos_emb(positions))
        for layer in self.layers:
            h = layer(h)
        return self.head(self.ln_f(h))

In [ ]:
%%writefile /kaggle/working/phase6_arch1/hub_files.py
"""Finds Phase-1 files: an attached Kaggle dataset first, then a local cache, then Hugging Face."""
import os
import re

from huggingface_hub import hf_hub_download

KAGGLE_INPUT = "/kaggle/input"


def _squash(text):
    return re.sub(r"[^a-z0-9]", "", text.lower())


def find_in_kaggle_input(filename, hint):
    """Only accepts a file whose path mentions the architecture (e.g. 'arch3', 'arch-3', 'Arch_3'),
    because arch 2, 3 and 4 all call their weights model_final.pt."""
    if not os.path.isdir(KAGGLE_INPUT):
        return None
    hits = []
    for root, _, files in os.walk(KAGGLE_INPUT):
        if filename in files:
            path = os.path.join(root, filename)
            if _squash(hint) in _squash(path):
                hits.append(path)
    return sorted(hits)[0] if hits else None


def fetch_from_hub(repo, filename, local_dir):
    try:
        return hf_hub_download(repo, filename, local_dir=local_dir, token=os.environ.get("HF_TOKEN"))
    except Exception:
        return None


def locate(filenames, repos, hint, local_dir):
    """Names are tried in order of preference. Returns (path, where it came from)."""
    for name in filenames:
        path = find_in_kaggle_input(os.path.basename(name), hint)
        if path:
            return path, "kaggle input"
    for name in filenames:
        for repo in repos:
            cached = os.path.join(local_dir, _squash(repo), name)
            if os.path.exists(cached):
                return cached, f"cache of hf://{repo}"
            path = fetch_from_hub(repo, name, os.path.join(local_dir, _squash(repo)))
            if path:
                return path, f"hf://{repo}"
    raise FileNotFoundError(f"None of {filenames} found under /kaggle/input (path must mention '{hint}') or in "
                            f"{repos}. Attach the Phase-1 weights as a Kaggle dataset, or check HF_TOKEN and the repo names.")

In [ ]:
%%writefile /kaggle/working/phase6_arch1/gen_core.py
"""
Phase 6 (abstractive summarization) machinery shared by all four architectures:
source selection, tokenized arrays, adapters, the decoder-only summarizer wrapper, decoding with length control and
redundancy control, ROUGE, and the submission writer. Each architecture's lm.py supplies the model-specific pieces.
"""
import os
import json
import math
from multiprocessing import Pool

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

IGNORE = -100


# ------------------------------------------------------------------------------------------- data
def load_split(path, with_summary):
    with open(path, encoding="utf-8") as f:
        rows = json.load(f)
    ids = [r["id"] for r in rows]
    docs = [r["document"] for r in rows]
    sums = [r["summary"] for r in rows] if with_summary else None
    return ids, docs, sums


_tok = None
_cfg = None


def _init_worker(cfg_dict):
    global _tok, _cfg
    import lm as LM
    _tok = LM.load_tokenizer()
    _cfg = cfg_dict


def select_source(ids, budget, head_frac, n_spread):
    """Keep the opening of the judgment (parties, facts) and evenly spaced pieces of the rest, ending at the last
    token - so the whole case is represented even when it is 50 times longer than the budget."""
    if len(ids) <= budget:
        return ids
    head = int(budget * head_frac)
    piece = (budget - head) // n_spread
    rest = len(ids) - head
    starts = [head + round(i * (rest - piece) / max(1, n_spread - 1)) for i in range(n_spread)]
    out = ids[:head]
    for s in starts:
        out = out + ids[s:s + piece]
    return out


def encode_example(pair):
    doc, summ = pair
    src = select_source(_tok.encode(doc[:_cfg["char_cap"]]), _cfg["src_len"], _cfg["head_frac"], _cfg["n_spread"])
    tgt = _tok.encode(summ)[:_cfg["tgt_len"]] if summ is not None else []
    return src, tgt, len(doc.split()), (len(summ.split()) if summ is not None else 0)


def prepare_split(docs, sums, cfg_dict, workers):
    pairs = list(zip(docs, sums if sums is not None else [None] * len(docs)))
    with Pool(workers, initializer=_init_worker, initargs=(cfg_dict,)) as pool:
        res = pool.map(encode_example, pairs, chunksize=16)
    S, T = cfg_dict["src_len"], cfg_dict["tgt_len"]
    src = np.full((len(res), S), cfg_dict["pad"], dtype=np.int32)
    tgt = np.full((len(res), T), cfg_dict["pad"], dtype=np.int32)
    src_len = np.zeros(len(res), dtype=np.int32)
    tgt_len = np.zeros(len(res), dtype=np.int32)
    doc_words = np.zeros(len(res), dtype=np.int32)
    sum_words = np.zeros(len(res), dtype=np.int32)
    for i, (s, t, dw, sw) in enumerate(res):
        src[i, :len(s)], src_len[i] = s, len(s)
        tgt[i, :len(t)], tgt_len[i] = t, len(t)
        doc_words[i], sum_words[i] = dw, sw
    return {"src": src, "src_len": src_len, "tgt": tgt, "tgt_len": tgt_len, "doc_words": doc_words, "sum_words": sum_words}


# ----------------------------------------------------------------------------------- adaptation
class LoRALinear(nn.Module):
    """Hu et al. 2021: W x + (alpha / r) B A x, B = 0 at the start."""

    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        self.scale = alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale


class LoRMALinear(nn.Module):
    """Bihany, Patel & Modi 2025 (LoRMA+): h = (alpha/r B A + I) W0 x, computed as W0x -> A -> B."""

    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        self.scale = alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        h = F.linear(x, self.base.weight)
        h = h + F.linear(F.linear(self.drop(h), self.A), self.B) * self.scale
        return h + self.base.bias if self.base.bias is not None else h


class Adapter(nn.Module):
    """Bottleneck adapter on a sub-layer's output: h + up(gelu(down(h))), up = 0 at the start.
    Houlsby et al. 2019 put one after attention and one after the feed-forward block;
    Pfeiffer et al. 2021 keep only the feed-forward one."""

    def __init__(self, base, bottleneck):
        super().__init__()
        self.base = base
        d = base.out_features
        self.down = nn.Linear(d, bottleneck)
        self.up = nn.Linear(bottleneck, d)
        nn.init.normal_(self.down.weight, std=0.02)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        h = self.base(x)
        return h + self.up(F.gelu(self.down(h)))


def wrap_linears(root, names, make, skip=("router",)):
    n = 0
    for parent in list(root.modules()):
        for name, child in list(parent.named_children()):
            if name in skip:
                continue
            if name in names and isinstance(child, nn.Linear):
                setattr(parent, name, make(child))
                n += 1
    return n


def configure_method(model, method, cfg):
    if method in ("full", "kd"):
        for p in model.parameters():
            p.requires_grad = True
        return "all weights trainable"
    for p in model.parameters():
        p.requires_grad = False
    if method == "lora":
        n = wrap_linears(model, cfg.ADAPTER_TARGETS, lambda m: LoRALinear(m, cfg.ADAPTER_RANK, cfg.LORA_ALPHA, cfg.ADAPTER_DROPOUT))
        return f"LoRA on {n} attention projections (r={cfg.ADAPTER_RANK})"
    if method == "lorma":
        n = wrap_linears(model, cfg.ADAPTER_TARGETS, lambda m: LoRMALinear(m, cfg.ADAPTER_RANK, cfg.LORMA_ALPHA, cfg.ADAPTER_DROPOUT))
        return f"LoRMA on {n} attention projections (r={cfg.ADAPTER_RANK})"
    if method == "houlsby":
        n = wrap_linears(model, cfg.HOULSBY_SITES, lambda m: Adapter(m, cfg.ADAPTER_BOTTLENECK))
        return f"{n} Houlsby adapters (after attention and feed-forward outputs, size {cfg.ADAPTER_BOTTLENECK})"
    if method == "pfeiffer":
        n = wrap_linears(model, cfg.PFEIFFER_SITES, lambda m: Adapter(m, cfg.ADAPTER_BOTTLENECK))
        return f"{n} Pfeiffer adapters (after the feed-forward output only, size {cfg.ADAPTER_BOTTLENECK})"
    if method == "prompt":
        # Lester et al. 2021: learned virtual tokens in front of the source, initialised from real token embeddings
        g = torch.Generator().manual_seed(cfg.SEED)
        w = model.tok_emb.weight
        pick = torch.randint(100, w.size(0) - 100, (cfg.PROMPT_LEN,), generator=g)
        model.prompt = nn.Parameter(w[pick].detach().clone())
        return f"{cfg.PROMPT_LEN} soft-prompt vectors in front of every source"
    raise ValueError(method)


def count_params(model):
    return sum(p.numel() for p in model.parameters()), sum(p.numel() for p in model.parameters() if p.requires_grad)


def trainable_state(model):
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().float().half() for k, v in model.state_dict().items() if k in names}


def build_optimizer(model, method, cfg):
    groups = {}
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name == "prompt":
            lr = cfg.PROMPT_LR
        elif method in ("full", "kd"):
            lr = cfg.FULL_LR
        else:
            lr = cfg.ADAPTER_LR
        wd = 0.0 if (p.ndim < 2 or name == "prompt") else cfg.WEIGHT_DECAY
        groups.setdefault((lr, wd), []).append(p)
    return torch.optim.AdamW([{"params": ps, "lr": lr, "weight_decay": wd} for (lr, wd), ps in groups.items()],
                             betas=(0.9, 0.98), eps=1e-6)


# ------------------------------------------------------------------- decoder-only summarizer
def decoder_batch(src, src_len, tgt, tgt_len, sep_id, pad_id, eos_id):
    """Right-padded [source] <sep> [summary] (<eos> if the summary was not cut) and next-token labels that are
    only set where the next token belongs to the summary."""
    rows, labels = [], []
    for s, sl, t, tl in zip(src, src_len, tgt, tgt_len):
        summary = list(t[:tl]) + ([eos_id] if tl < len(t) else [])
        seq = list(s[:sl]) + [sep_id] + summary
        lab = [IGNORE] * sl + summary + [IGNORE]
        rows.append(seq)
        labels.append(lab[:len(seq)])
    L = max(map(len, rows))
    ids = torch.full((len(rows), L), pad_id, dtype=torch.long)
    lab = torch.full((len(rows), L), IGNORE, dtype=torch.long)
    for i, (r, l) in enumerate(zip(rows, labels)):
        ids[i, :len(r)] = torch.tensor(r)
        lab[i, :len(l)] = torch.tensor(l)
    return ids, lab


class DecoderLM(nn.Module):
    """Wraps a Phase-1 decoder-only model. Subclasses (lm.py) provide add_positions() and run_blocks()."""
    is_encoder_decoder = False

    def __init__(self):
        super().__init__()
        self.prompt = None

    def _embed(self, ids, valid):
        x = self.tok_emb(ids)
        if self.prompt is not None:
            P = self.prompt.size(0)
            x = torch.cat([self.prompt.to(x.dtype).unsqueeze(0).expand(x.size(0), -1, -1), x], dim=1)
            valid = torch.cat([valid.new_ones(valid.size(0), P), valid], dim=1)
        pos = (valid.long().cumsum(1) - 1).clamp(min=0)
        return x, valid, pos

    def teacher_forced(self, batch, device):
        ids, labels = batch
        ids, labels = ids.to(device), labels.to(device)
        x, valid, pos = self._embed(ids, torch.ones_like(ids, dtype=torch.bool))
        x = self.add_positions(x, pos)
        h, _ = self.run_blocks(x, pos, None, None)
        h = self.final_norm(h)[:, h.size(1) - ids.size(1):]
        keep = labels != IGNORE
        logits = F.linear(h[keep], self.tok_emb.weight)
        return logits.float(), labels[keep]

    def make_batch(self, src, src_len, tgt, tgt_len, tok):
        return decoder_batch(src, src_len, tgt, tgt_len, tok.sep_id, tok.pad_id, tok.eos_id)

    @torch.no_grad()
    def generate(self, sources, min_new, max_new, tok, cfg, device):
        """sources: list of token lists. Left-padded batch; every row stops on its own length budget."""
        B = len(sources)
        rows = [list(s) + [tok.sep_id] for s in sources]
        L = max(map(len, rows))
        ids = torch.full((B, L), tok.pad_id, dtype=torch.long)
        valid = torch.zeros((B, L), dtype=torch.bool)
        for i, r in enumerate(rows):
            ids[i, L - len(r):] = torch.tensor(r)
            valid[i, L - len(r):] = True
        ids, valid = ids.to(device), valid.to(device)
        x, valid, pos = self._embed(ids, valid)
        x = self.add_positions(x, pos)
        S = x.size(1)
        causal = torch.tril(torch.ones(S, S, dtype=torch.bool, device=device))
        mask = (causal[None, None] & valid[:, None, None, :]) | torch.eye(S, dtype=torch.bool, device=device)[None, None]
        h, caches = self.run_blocks(x, pos, None, mask)
        logits = F.linear(self.final_norm(h[:, -1:]), self.tok_emb.weight)[:, -1].float()
        next_pos = pos[:, -1] + 1

        state = DecodeState(B, logits.size(-1), min_new, max_new, tok, cfg, device)
        while not state.done():
            nxt = state.step(logits)
            valid = torch.cat([valid, torch.ones((B, 1), dtype=torch.bool, device=device)], dim=1)
            x = self.add_positions(self.tok_emb(nxt[:, None]), next_pos[:, None])
            h, caches = self.run_blocks(x, next_pos[:, None], caches, valid[:, None, None, :])
            logits = F.linear(self.final_norm(h), self.tok_emb.weight)[:, -1].float()
            next_pos = next_pos + 1
        return state.outputs()


# ---------------------------------------------------------------------------------- decoding
class DecodeState:
    """Greedy decoding with a length budget per row, a repetition penalty (Keskar et al. 2019) and no repeated
    n-grams - the redundancy controls the summaries need."""

    def __init__(self, B, V, min_new, max_new, tok, cfg, device):
        self.B, self.tok, self.cfg = B, tok, cfg
        self.min_new = torch.tensor(min_new, device=device)
        self.max_new = torch.tensor(max_new, device=device)
        self.seen = torch.zeros((B, V), dtype=torch.bool, device=device)
        self.out = [[] for _ in range(B)]
        self.ngrams = [dict() for _ in range(B)]
        self.finished = torch.zeros(B, dtype=torch.bool, device=device)
        self.t = 0

    def done(self):
        return bool(self.finished.all())

    def step(self, logits):
        cfg, tok = self.cfg, self.tok
        logits = logits.clone()
        logits[:, tok.pad_id] = float("-inf")
        for name in ("unk_id", "bos_id"):                # arch 1's word vocabulary has these; decoding would drop them
            if hasattr(tok, name):
                logits[:, getattr(tok, name)] = float("-inf")
        logits[self.t < self.min_new, tok.eos_id] = float("-inf")
        if cfg.REPETITION_PENALTY != 1.0:
            pen = torch.where(logits > 0, logits / cfg.REPETITION_PENALTY, logits * cfg.REPETITION_PENALTY)
            logits = torch.where(self.seen, pen, logits)
        n = cfg.NO_REPEAT_NGRAM
        if n > 1:
            for i in range(self.B):
                if not self.finished[i] and len(self.out[i]) >= n - 1:
                    banned = self.ngrams[i].get(tuple(self.out[i][-(n - 1):]))
                    if banned:
                        logits[i, list(banned)] = float("-inf")
        nxt = logits.argmax(-1)
        nxt = torch.where(self.finished, torch.full_like(nxt, tok.pad_id), nxt)
        for i, t in enumerate(nxt.tolist()):
            if self.finished[i]:
                continue
            seq = self.out[i]
            seq.append(t)
            if n > 1 and len(seq) >= n:
                self.ngrams[i].setdefault(tuple(seq[-n:-1]), set()).add(seq[-1])
        self.seen[torch.arange(self.B), nxt] = True
        self.t += 1
        self.finished |= (nxt == tok.eos_id) | (self.t >= self.max_new)
        return nxt

    def outputs(self):
        return [[t for t in seq if t not in (self.tok.eos_id, self.tok.pad_id)] for seq in self.out]


def length_budget(doc_words, cfg, tokens_per_word, ceiling):
    """Summary length follows document length (the training median ratio), clipped to sensible bounds."""
    words = np.clip(np.asarray(doc_words) / cfg.LENGTH_RATIO, cfg.MIN_SUMMARY_WORDS, cfg.MAX_SUMMARY_WORDS)
    target = words * tokens_per_word
    max_new = np.minimum(np.ceil(target * (1 + cfg.LENGTH_SLACK)), ceiling).astype(int)
    min_new = np.minimum(np.floor(target * (1 - cfg.LENGTH_SLACK)), max_new - 1).astype(int)
    return min_new.tolist(), max_new.tolist()


def tidy(text, sentence_case):
    text = " ".join(text.split())
    if sentence_case:                                    # arch 1 generates lower-case words
        out, cap = [], True
        for w in text.split(" "):
            out.append(w[:1].upper() + w[1:] if cap and w[:1].isalpha() else w)
            cap = w.endswith((".", "?", "!"))
        text = " ".join(out)
    return text


# ------------------------------------------------------------------------------------------ ROUGE
def _lcs(a, b):
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b):
            cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    return prev[-1]


def rouge(ref, hyp):
    r, h = ref.lower().split(), hyp.lower().split()
    if not r or not h:
        return 0.0, 0.0
    from collections import Counter
    overlap = sum((Counter(r) & Counter(h)).values())
    p1, r1 = overlap / len(h), overlap / len(r)
    f1 = 2 * p1 * r1 / (p1 + r1) if p1 + r1 else 0.0
    lcs = _lcs(r, h)
    pl, rl = lcs / len(h), lcs / len(r)
    fl = 2 * pl * rl / (pl + rl) if pl + rl else 0.0
    return f1, fl


def write_submission(ids, summaries, out_dir, zip_name):
    import zipfile
    assert len(ids) == len(summaries) == len(set(map(str, ids)))
    payload = [{"id": i, "summary": s if s.strip() else "."} for i, s in zip(ids, summaries)]
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, "prediction.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False)
    zip_path = os.path.join(out_dir, zip_name)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(path, arcname="prediction.json")
    with zipfile.ZipFile(zip_path) as z:
        assert z.namelist() == ["prediction.json"]
    return path, zip_path

In [ ]:
%%writefile /kaggle/working/phase6_arch1/lm.py
"""Arch 1 (GPT-2 style decoder, word-level vocabulary) as a summarizer.
Phase 1 never needed a KV cache, so the attention step below re-uses the block's own projections with one; adapters
wrapped around those projections therefore apply during generation too."""
import json
import os

import torch
import torch.nn as nn
import torch.nn.functional as F

import config as C
from hub_files import locate, fetch_from_hub  # noqa: F401  (fetch_from_hub is used by run_sum)
from phase1_model import GPT2LM
from tokenizer import NerTokenizer
from gen_core import DecoderLM


def load_tokenizer():
    path, _ = locate(C.TOKENIZER_FILES, C.TOKENIZER_REPOS, C.KAGGLE_HINT, C.CACHE_DIR)
    tok = NerTokenizer()
    tok.load(path)
    tok.sep_id = tok.eos_id
    return tok


def attention(attn, x, past, key_mask, training):
    B, T, D = x.shape
    qkv = attn.W_qkv(x).reshape(B, T, 3, attn.n_head, attn.head_dim).permute(2, 0, 3, 1, 4)
    q, k, v = qkv[0], qkv[1], qkv[2]
    if past is not None:
        k = torch.cat([past[0], k], dim=2)
        v = torch.cat([past[1], v], dim=2)
    p = attn.dropout_p if training else 0.0
    if key_mask is None:
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=p)
    else:
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=key_mask, dropout_p=p)
    out = out.permute(0, 2, 1, 3).reshape(B, T, D)
    return attn.resid_drop(attn.W_o(out)), (k, v)


class Arch1LM(DecoderLM):
    def __init__(self, gpt, keep_layers):
        super().__init__()
        self.tok_emb = gpt.token_emb
        self.pos_emb = gpt.pos_emb
        self.emb_drop = gpt.emb_drop
        self.layers = nn.ModuleList(gpt.layers if keep_layers is None else [gpt.layers[i] for i in keep_layers])
        self.final_norm = gpt.ln_f
        self.max_len = gpt.pos_emb.num_embeddings

    def add_positions(self, x, pos):
        return self.emb_drop(x + self.pos_emb(pos))      # learned absolute positions, per row

    def run_blocks(self, x, pos, caches, key_mask):
        new = []
        for i, blk in enumerate(self.layers):
            a, present = attention(blk.attn, blk.ln1(x), None if caches is None else caches[i], key_mask, self.training)
            x = x + a
            x = x + blk.ffn(blk.ln2(x))
            new.append(present)
        return x, new


def build_lm(keep_layers=None):
    cfg_path, _ = locate(C.CONFIG_FILES, C.PHASE1_REPOS, C.KAGGLE_HINT, C.CACHE_DIR)
    path, source = locate(C.WEIGHT_FILES, C.PHASE1_REPOS, C.KAGGLE_HINT, C.CACHE_DIR)
    with open(cfg_path) as f:
        cfg = json.load(f)
    gpt = GPT2LM(cfg["vocab_size"], cfg["d_model"], cfg["n_head"], cfg["n_layer"], cfg["d_ff"], cfg["dropout"], cfg["max_len"])
    gpt.load_state_dict({k.replace("_orig_mod.", ""): v.float() for k, v in torch.load(path, map_location="cpu").items()})
    if os.environ.get("P2_VERBOSE", "1") == "1":
        print(f"Phase-1 arch 1 weights: {path} ({source})", flush=True)
    return Arch1LM(gpt, keep_layers)


def student_layers():
    return list(C.STUDENT_LAYERS)

In [ ]:
%%writefile /kaggle/working/phase6_arch1/run_sum.py
"""
One Phase-6 experiment = one call:

    python run_sum.py --method lorma

Trains for a fixed wall-clock budget (the best checkpoint is chosen by teacher-forced loss on held-out documents),
generates summaries for some held-out documents to measure ROUGE-1 / ROUGE-L and length, then summarizes the test set
and pushes checkpoint, results and submission zip to Hugging Face.
"""
import os
import sys
import json
import math
import time
import random
import argparse

import numpy as np
import torch
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import config as C
import lm as LM
from gen_core import (configure_method, count_params, trainable_state, build_optimizer, length_budget, tidy, rouge,
                      write_submission)


def log(*a):
    print(*a, flush=True)


def build_model(method, keep_layers=None):
    model = LM.build_lm(keep_layers)
    note = configure_method(model, method, C)
    return model, note


def load_finetuned(method, device):
    path = os.path.join(C.OUT_DIR, method, f"sum_{method}.pt")
    if not os.path.exists(path):
        path = LM.fetch_from_hub(C.SUM_REPO, f"{method}/sum_{method}.pt", os.path.join(C.OUT_DIR, "_hub"))
    if path is None or not os.path.exists(path):
        raise FileNotFoundError(f"No fine-tuned '{method}' model locally or on {C.SUM_REPO}; run '{method}' first.")
    blob = torch.load(path, map_location="cpu")
    model, _ = build_model(method, blob.get("keep_layers"))
    _, unexpected = model.load_state_dict({k: v.float() for k, v in blob["weights"].items()}, strict=False)
    assert not unexpected, unexpected
    return model.to(device).eval()


def push(files, message):
    if not os.environ.get("HF_TOKEN"):
        return
    try:
        from huggingface_hub import HfApi
        api = HfApi()
        api.create_repo(C.SUM_REPO, private=True, exist_ok=True, token=os.environ["HF_TOKEN"])
        for local, remote in files:
            api.upload_file(path_or_fileobj=local, path_in_repo=remote, repo_id=C.SUM_REPO,
                            token=os.environ["HF_TOKEN"], commit_message=message)
        log(f"pushed to {C.SUM_REPO}: {message}")
    except Exception as e:
        log(f"HF push failed ({type(e).__name__}: {e}) - the files are still in {C.OUT_DIR}")


def load_npz(name):
    z = np.load(os.path.join(C.DATA_DIR, f"{name}.npz"))
    return {k: z[k] for k in z.files}


@torch.no_grad()
def dev_loss(model, d, idx, tok, device):
    model.eval()
    total, n = 0.0, 0
    for b in range(0, len(idx), C.MICRO_DOCS):
        j = idx[b:b + C.MICRO_DOCS]
        batch = model.make_batch(d["src"][j], d["src_len"][j], d["tgt"][j], d["tgt_len"][j], tok)
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            logits, gold = model.teacher_forced(batch, device)
        total += F.cross_entropy(logits, gold, reduction="sum").item()
        n += gold.numel()
    return total / max(1, n)


def summarize(model, d, idx, tok, device, tokens_per_word):
    srcs = [d["src"][i][:d["src_len"][i]].tolist() for i in idx]
    ceiling = C.MAX_NEW_TOKENS
    min_new, max_new = length_budget(d["doc_words"][idx], C, tokens_per_word, ceiling)
    out = [None] * len(idx)
    order = np.argsort([len(s) for s in srcs])                    # similar lengths together -> little padding
    for b in range(0, len(order), C.GEN_BATCH):
        part = order[b:b + C.GEN_BATCH]
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            toks = model.generate([srcs[i] for i in part], [min_new[i] for i in part], [max_new[i] for i in part],
                                  tok, C, device)
        for i, t in zip(part, toks):
            out[i] = tidy(tok.decode(t), C.SENTENCE_CASE)
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--method", required=True, choices=["full", "lorma", "lora", "houlsby", "pfeiffer", "prompt", "kd"])
    method = ap.parse_args().method
    random.seed(C.SEED)
    np.random.seed(C.SEED)
    torch.manual_seed(C.SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out_dir = os.path.join(C.OUT_DIR, method)
    os.makedirs(out_dir, exist_ok=True)
    t0 = time.time()

    tok = LM.load_tokenizer()
    train, test = load_npz("train"), load_npz("test")
    meta = json.load(open(os.path.join(C.DATA_DIR, "meta.json")))
    if C.LENGTH_RATIO == "auto":
        C.LENGTH_RATIO = meta["length_ratio"]          # median document/summary word ratio of the training set
    n = len(train["src_len"])
    perm = np.random.RandomState(C.SEED).permutation(n)
    dev_idx, tr_idx = np.sort(perm[:C.DEV_DOCS]), perm[C.DEV_DOCS:]
    if C.TRAIN_DOC_LIMIT:
        tr_idx = tr_idx[:C.TRAIN_DOC_LIMIT]
    loss_idx, gen_idx = dev_idx[:C.DEV_LOSS_DOCS], dev_idx[:C.DEV_GEN_DOCS]
    log(f"[{method}] train {len(tr_idx):,} docs | held-out {len(dev_idx)} | test {len(test['src_len'])} | "
        f"source {int(train['src_len'].mean())} tokens, summary {int(train['tgt_len'].mean())} tokens on average")

    keep = LM.student_layers() if method == "kd" else None
    model, note = build_model(method, keep)
    model.to(device)
    total, trainable = count_params(model)
    log(f"[{method}] {note} | params {total / 1e6:.1f}M, trainable {trainable / 1e6:.2f}M | budget {C.TRAIN_MINUTES} min")

    teacher = None
    if method == "kd":
        teacher = load_finetuned(C.KD_TEACHER, device)
        log(f"[kd] teacher = fine-tuned '{C.KD_TEACHER}' ({count_params(teacher)[0] / 1e6:.1f}M) -> student keeping "
            f"layers {keep} ({total / 1e6:.1f}M); it learns the teacher's next-token distributions (word-level KD)")

    optimizer = build_optimizer(model, method, C)
    base_lrs = [g["lr"] for g in optimizer.param_groups]
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    docs_per_step = C.MICRO_DOCS * C.GRAD_ACCUM
    step_cap = max(1, C.MAX_EPOCHS * len(tr_idx) // docs_per_step)
    budget = C.TRAIN_MINUTES * 60
    order = np.random.RandomState(C.SEED).permutation(tr_idx)
    cursor, step, seen, running = 0, 0, 0, 0.0
    best, history = {"loss": float("inf"), "step": 0}, []
    t_train = time.time()

    def progress():
        return max(step / step_cap, (time.time() - t_train) / budget)

    while progress() < 1.0:
        warm = min(1.0, (step + 1) / C.WARMUP_STEPS)
        lr_scale = warm * (C.MIN_LR_RATIO + (1 - C.MIN_LR_RATIO) * 0.5 * (1 + math.cos(math.pi * min(1.0, progress()))))
        for g, lr in zip(optimizer.param_groups, base_lrs):
            g["lr"] = lr * lr_scale
        model.train()
        for _ in range(C.GRAD_ACCUM):
            if cursor + C.MICRO_DOCS > len(order):
                order = np.random.RandomState(C.SEED + step).permutation(tr_idx)
                cursor = 0
            j = np.sort(order[cursor:cursor + C.MICRO_DOCS])
            cursor += C.MICRO_DOCS
            batch = model.make_batch(train["src"][j], train["src_len"][j], train["tgt"][j], train["tgt_len"][j], tok)
            with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
                logits, gold = model.teacher_forced(batch, device)
                loss = F.cross_entropy(logits, gold, label_smoothing=C.LABEL_SMOOTHING)
                if teacher is not None:
                    with torch.no_grad():
                        t_logits, _ = teacher.teacher_forced(batch, device)
                    T = C.KD_TEMPERATURE
                    kl = F.kl_div(F.log_softmax(logits / T, -1), F.softmax(t_logits.float() / T, -1),
                                  reduction="batchmean") * T * T
                    loss = C.KD_ALPHA * loss + (1 - C.KD_ALPHA) * kl
            scaler.scale(loss / C.GRAD_ACCUM).backward()
            running += loss.item()
            seen += 1
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_([q for q in model.parameters() if q.requires_grad], 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        step += 1

        if step % C.EVAL_EVERY_STEPS == 0 or progress() >= 1.0:
            dl = dev_loss(model, train, loss_idx, tok, device)
            history.append({"step": step, "docs_seen": step * docs_per_step, "train_loss": running / max(1, seen),
                            "dev_loss": dl, "minutes": (time.time() - t0) / 60})
            log(f"[{method}] step {step} | {step * docs_per_step:,} docs seen | loss {running / max(1, seen):.4f} | "
                f"held-out loss {dl:.4f} (ppl {math.exp(min(dl, 20)):.1f}) | {100 * min(1.0, progress()):.0f}% of budget")
            running, seen = 0.0, 0
            if dl < best["loss"]:
                best = {"loss": dl, "step": step}
                torch.save({"method": method, "keep_layers": keep, "weights": trainable_state(model), "step": step},
                           os.path.join(out_dir, f"sum_{method}.pt"))

    # --------------------------------------------- best checkpoint: summaries for held-out docs, then the test set
    blob = torch.load(os.path.join(out_dir, f"sum_{method}.pt"), map_location="cpu")
    model.load_state_dict({k: v.float() for k, v in blob["weights"].items()}, strict=False)
    model.eval()
    del teacher
    if device.type == "cuda":
        torch.cuda.empty_cache()

    with open(os.path.join(C.DATA_DIR, "train_summaries.json")) as f:
        references = json.load(f)
    t_gen = time.time()
    dev_out = summarize(model, train, gen_idx, tok, device, meta["tokens_per_word"])
    gen_seconds = time.time() - t_gen
    pairs = [rouge(references[int(i)], h) for i, h in zip(gen_idx, dev_out)]
    r1 = float(np.mean([p[0] for p in pairs]))
    rl = float(np.mean([p[1] for p in pairs]))
    words = [len(h.split()) for h in dev_out]
    ref_words = [len(references[int(i)].split()) for i in gen_idx]

    test_out = summarize(model, test, np.arange(len(test["src_len"])), tok, device, meta["tokens_per_word"])
    with open(os.path.join(C.DATA_DIR, "test_ids.json")) as f:
        test_ids = json.load(f)
    _, zip_path = write_submission(test_ids, test_out, out_dir, f"submission_{C.ARCH}_{method}.zip")
    with open(os.path.join(out_dir, "dev_samples.json"), "w") as f:
        json.dump([{"reference": references[int(i)][:1500], "generated": h[:1500]} for i, h in zip(gen_idx[:5], dev_out[:5])],
                  f, indent=1, ensure_ascii=False)

    summary = {
        "arch": C.ARCH, "method": method, "best_step": best["step"], "heldout_loss": best["loss"],
        "heldout_ppl": math.exp(min(best["loss"], 20)), "dev_rouge1_f1": r1, "dev_rougeL_f1": rl,
        "dev_words_generated": float(np.mean(words)), "dev_words_reference": float(np.mean(ref_words)),
        "test_words_generated": float(np.mean([len(s.split()) for s in test_out])),
        "params_total": total, "params_trainable": trainable,
        "seconds_per_summary": round(gen_seconds / max(1, len(gen_idx)), 2),
        "peak_gpu_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if device.type == "cuda" else 0.0,
        "minutes": round((time.time() - t0) / 60, 1), "docs_seen": step * docs_per_step, "history": history,
    }
    with open(os.path.join(out_dir, "results.json"), "w") as f:
        json.dump(summary, f, indent=2)
    log(f"[{method}] DONE | held-out loss {best['loss']:.4f} | ROUGE-1 {r1:.4f} ROUGE-L {rl:.4f} on {len(gen_idx)} held-out docs | "
        f"{np.mean(words):.0f} words generated vs {np.mean(ref_words):.0f} in references -> {zip_path}")

    push([(os.path.join(out_dir, f"sum_{method}.pt"), f"{method}/sum_{method}.pt"),
          (os.path.join(out_dir, "results.json"), f"{method}/results.json"),
          (os.path.join(out_dir, "dev_samples.json"), f"{method}/dev_samples.json"),
          (zip_path, f"{method}/{os.path.basename(zip_path)}")], f"{C.ARCH} {method}: ROUGE-L {rl:.4f}")


if __name__ == "__main__":
    main()

## 4. Data: download, select the source, tokenize
Done once with all CPU cores and stored as compact arrays.

In [ ]:
import importlib
import config as C
importlib.reload(C)
import lm as LM
import gen_core as G
importlib.reload(G)

for split, folder in (("train", "Train"), ("test", "Test")):
    hf_hub_download(C.HF_DATASET_REPO, f"Phase_6_Summ/{folder}/{split}.json", repo_type="dataset",
                    local_dir=C.DATA_DIR, token=HF_TOKEN)

tok = LM.load_tokenizer()                       # caches the tokenizer before the worker processes start
prep_cfg = {"src_len": C.SRC_LEN, "tgt_len": C.TGT_LEN, "char_cap": C.CHAR_CAP, "head_frac": C.HEAD_FRAC,
            "n_spread": C.N_SPREAD, "pad": tok.pad_id}

ids, docs, sums = G.load_split(C.RAW["train"], with_summary=True)
arrays = G.prepare_split(docs, sums, prep_cfg, C.PREP_WORKERS)
np.savez(os.path.join(C.DATA_DIR, "train.npz"), **arrays)
json.dump(sums, open(os.path.join(C.DATA_DIR, "train_summaries.json"), "w"))
full = arrays["tgt_len"] < C.TGT_LEN
meta = {"tokens_per_word": float(np.median(arrays["tgt_len"][full] / np.maximum(arrays["sum_words"][full], 1))),
        "length_ratio": float(np.median(arrays["doc_words"] / np.maximum(arrays["sum_words"], 1)))}
json.dump(meta, open(os.path.join(C.DATA_DIR, "meta.json"), "w"))
print(f"train: {len(ids):,} docs | source {arrays['src_len'].mean():.0f} tokens | summary {arrays['tgt_len'].mean():.0f} tokens "
      f"({100 * (~full).mean():.0f}% cut at {C.TGT_LEN}) | {meta['tokens_per_word']:.2f} tokens/word | "
      f"document/summary ratio {meta['length_ratio']:.2f}")
del docs, sums, arrays
gc.collect()

ids, docs, _ = G.load_split(C.RAW["test"], with_summary=False)
arrays = G.prepare_split(docs, None, prep_cfg, C.PREP_WORKERS)
np.savez(os.path.join(C.DATA_DIR, "test.npz"), **arrays)
json.dump(ids, open(os.path.join(C.DATA_DIR, "test_ids.json"), "w"))
print(f"test: {len(ids)} docs | source {arrays['src_len'].mean():.0f} tokens | document words median {np.median(arrays['doc_words']):.0f}")

## 5. Load the Phase-1 weights once
A quick generation from the untouched Phase-1 model checks the whole decoding path and caches the files.

In [ ]:
probe = LM.build_lm().eval()
test = np.load(os.path.join(C.DATA_DIR, "test.npz"))
with torch.no_grad():
    out = probe.generate([test["src"][0][:200].tolist()], [10], [20], tok, C, torch.device("cpu"))
print(f"model ready: {sum(p.numel() for p in probe.parameters()) / 1e6:.1f}M parameters | sample continuation: {tok.decode(out[0])!r}")
del probe, test

## 6. Run the experiments
Each experiment is its own process (`run_sum.py --method ...`); with two GPUs, two run at once, so the six take about
three rounds of `TRAIN_MINUTES` + ~15 minutes (training, then summarising 40 held-out and 100 test documents).
`kd` starts as soon as `full` (its teacher) is done. Results, checkpoints and zips are pushed to `D-240357/legal-sum-arch1`.

In [ ]:
import subprocess, threading

pending = list(C.METHODS)
done, failed = set(), set()
lock = threading.Lock()


def next_job():
    with lock:
        if "kd" in pending and C.KD_TEACHER in failed:
            pending.remove("kd")
            print(f"skipping kd: its teacher '{C.KD_TEACHER}' failed")
        for m in pending:
            if m != "kd" or C.KD_TEACHER in done or C.KD_TEACHER not in C.METHODS:
                pending.remove(m)
                return m
        return "wait" if pending else None


def worker(gpu):
    while True:
        job = next_job()
        if job is None:
            return
        if job == "wait":
            time.sleep(15)
            continue
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
        proc = subprocess.Popen([sys.executable, "run_sum.py", "--method", job], cwd=WORK_DIR, env=env,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(f"[gpu{gpu}] {line}", end="")
        with lock:
            (done if proc.wait() == 0 else failed).add(job)


t0 = time.time()
threads = [threading.Thread(target=worker, args=(g,)) for g in range(max(1, torch.cuda.device_count()))]
for t in threads:
    t.start()
for t in threads:
    t.join()
print(f"\nfinished in {(time.time() - t0) / 60:.0f} min | ok: {sorted(done)} | failed: {sorted(failed)}")

## 7. Results
Held-out ROUGE-1 / ROUGE-L, length against the references, and cost. A generated summary is shown next to its reference.

In [ ]:
import pandas as pd

rows = []
for path in sorted(glob.glob(os.path.join(C.OUT_DIR, "*", "results.json"))):
    r = json.load(open(path))
    rows.append({"method": r["method"], "ROUGE-1": r["dev_rouge1_f1"], "ROUGE-L": r["dev_rougeL_f1"],
                 "held-out loss": r["heldout_loss"], "words (gen)": r["dev_words_generated"],
                 "words (ref)": r["dev_words_reference"], "docs seen": r["docs_seen"],
                 "params (M)": r["params_total"] / 1e6, "trainable %": 100 * r["params_trainable"] / r["params_total"],
                 "s / summary": r["seconds_per_summary"], "minutes": r["minutes"], "peak GPU GB": r["peak_gpu_gb"]})
table = pd.DataFrame(rows).sort_values("ROUGE-L", ascending=False).round(4)
display(table)
best = table.iloc[0]["method"]
sample = json.load(open(os.path.join(C.OUT_DIR, best, "dev_samples.json")))[0]
print(f"--- reference ---\n{sample['reference'][:800]}\n\n--- {best} ---\n{sample['generated'][:800]}")

## 8. Pick a submission
Every experiment wrote `runs/<method>/submission_<arch>_<method>.zip`: a single `prediction.json` at the root holding a
JSON list of `{"id": ..., "summary": ...}` for all 100 test documents. The best on held-out ROUGE-L is also copied to
`/kaggle/working/submission.zip`. Upload the zips exactly as they are.

In [ ]:
shutil.copy(glob.glob(os.path.join(C.OUT_DIR, best, "submission_*.zip"))[0], "/kaggle/working/submission.zip")
print(f"best on held-out ROUGE-L: {best} -> /kaggle/working/submission.zip")
for z in sorted(glob.glob(os.path.join(C.OUT_DIR, "*", "submission_*.zip"))):
    print("  ", z)